In [1]:
import polars as pl

df = pl.read_parquet(
    "datasets/dataset_merged_with_families.parquet",
)

df


original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,cath_dominant,cath_all,cath_class,cath_arch,cath_topology,cath_homology
str,str,str,f64,bool,str,str,str,str,str,str,str
"""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""T71I""",-0.035821,false,"""megascale""","""G3DSA:3.30.1370.10""","""G3DSA:3.30.1370.10""","""G3DSA:3""","""30""","""1370""","""10"""
"""SAGGSAGGSAGGKVTIVVENIKVFGEDGKL…","""SAGGSAGGSAGGKVTIVVENIKVFGEDGKL…","""E33R""",-0.088899,false,"""megascale""","""G3DSA:2.40.50.140""","""G3DSA:2.40.50.140;G3DSA:6.20.3…","""G3DSA:2""","""40""","""50""","""140"""
"""SAGKMTGIVKWFNADKGFGFITPDDGSKDV…","""SAGKMTGIVKWFNADKGFGPITPDDGSKDV…","""F20P""",-0.216443,false,"""megascale""","""G3DSA:3.30.170.10""","""G3DSA:3.30.170.10""","""G3DSA:3""","""30""","""170""","""10"""
"""SAGGMIINNLKLIREKKKISQSELAALLES…","""SAGGMIIKNLKLIREKKKISQSELAALLES…","""N8K""",-0.262815,false,"""megascale""","""G3DSA:1.10.30.10""","""G3DSA:1.10.30.10""","""G3DSA:1""","""10""","""30""","""10"""
"""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""SSGGSSILDRAVIEHNLLSASKLYNNITFE…","""Q49S:E53H""",-0.12508,false,"""megascale""","""G3DSA:3.30.1370.10""","""G3DSA:3.30.1370.10""","""G3DSA:3""","""30""","""1370""","""10"""
…,…,…,…,…,…,…,…,…,…,…,…
"""SAGGSGPGLTDLFKTEKAAVKKMAKAIMAD…","""SAGGSGPGLTDLFKTEKAAVKKMAKAIMAD…","""Y38P:Y73E""",0.395344,true,"""megascale""","""G3DSA:1.10.30.10""","""G3DSA:1.10.30.10""","""G3DSA:1""","""10""","""30""","""10"""
"""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRV…","""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRE…","""E30V:Y50G""",0.067857,true,"""megascale""","""G3DSA:1.10.30.10""","""G3DSA:1.10.30.10""","""G3DSA:1""","""10""","""30""","""10"""
"""SAGGSAGGSAGGGHYRIRINGREFEIRGIS…","""SAGGSAGGSAGGGHYRIRINGREFEIRGIS…","""R42I""",0.007592,true,"""megascale""","""G3DSA:3.10.20.710""","""G3DSA:1.10.10.60;G3DSA:1.10.26…","""G3DSA:3""","""10""","""20""","""710"""


In [2]:
import polars as pl
import random
import os

# --- NASTAVENÍ ---
# df = ... (Váš načtený DataFrame)

HOMOLOGY_COL = "cath_dominant"
SEED = 42
BASE_DIR = "datasets"
FILE_PREFIX = "dataset_homology_split_"
FULL_PREFIX = os.path.join(BASE_DIR, FILE_PREFIX)

# Přejmenování na standardní názvy (zachováme celé sekvence)
RENAME_MAP = {
    "original_seq_full": "wt_sequence",
    "mutated_seq_full": "mut_sequence",
    "target": "fitness",
    "mut_type": "mutation",
    "cath_dominant": "cluster_id"
}

os.makedirs(BASE_DIR, exist_ok=True)
print("Zahajuji deterministický split (zachování celých sekvencí)...")

# 1. Příprava rodin (Deterministická)
family_counts = (
    df.group_by(HOMOLOGY_COL)
    .len()
    .sort(HOMOLOGY_COL)  # Nutné pro determinismus
    .to_dicts()
)

random.seed(SEED)
random.shuffle(family_counts)

# 2. Split Train vs Holdout (80/20 rodin)
total_rows = len(df)
holdout_goal = int(total_rows * 0.2)
train_fams, holdout_fams = [], []
curr_holdout = 0

for fam in family_counts:
    if curr_holdout < holdout_goal:
        holdout_fams.append(fam[HOMOLOGY_COL])
        curr_holdout += fam["len"]
    else:
        train_fams.append(fam[HOMOLOGY_COL])

# 3. Vytvoření Datasetů
train_df = df.filter(pl.col(HOMOLOGY_COL).is_in(train_fams))
holdout_df = df.filter(pl.col(HOMOLOGY_COL).is_in(holdout_fams))

# 4. Split Holdout na Val/Test (1:1)
# Seřadíme před mícháním pro determinismus
holdout_df = holdout_df.sort([HOMOLOGY_COL, "original_seq_full"])
holdout_df = holdout_df.sample(fraction=1.0, shuffle=True, seed=SEED)

split_point = len(holdout_df) // 2
val_df = holdout_df.slice(0, split_point)
test_df = holdout_df.slice(split_point, len(holdout_df) - split_point)

selected_column = ["wt_sequence", "mut_sequence", "mutation", "fitness", "cath_class", "cath_arch", "cath_topology",
                   "cath_homology", "data_source", "reverse"]


# 5. Uložení s přejmenováním
def save(d, name):
    valid_map = {k: v for k, v in RENAME_MAP.items() if k in d.columns}
    d.rename(valid_map).select(selected_column).write_csv(f"{FULL_PREFIX}{name}.csv")


def count_fams(d): return d[HOMOLOGY_COL].n_unique()


print(f"{'TRAIN':<10} | {len(train_df):>8} | {count_fams(train_df):>8} | {len(train_df) / total_rows * 100:>5.1f}%")
print(f"{'VAL':<10} | {len(val_df):>8} | {count_fams(val_df):>8} | {len(val_df) / total_rows * 100:>5.1f}%")
print(f"{'TEST':<10} | {len(test_df):>8} | {count_fams(test_df):>8} | {len(test_df) / total_rows * 100:>5.1f}%")

save(train_df, "train")
save(val_df, "validation")
save(test_df, "test")

if "reverse" in df.columns:
    save(test_df.filter(pl.col("reverse") == False), "test_noreverse")

print(f"Hotovo. Data s PLNOU délkou uložena do {BASE_DIR}")

Zahajuji deterministický split (zachování celých sekvencí)...
TRAIN      |  1361598 |      139 |  78.3%
VAL        |   188631 |       45 |  10.8%
TEST       |   188632 |       45 |  10.8%
Hotovo. Data s PLNOU délkou uložena do datasets


In [3]:
train_df

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,cath_dominant,cath_all,cath_class,cath_arch,cath_topology,cath_homology
str,str,str,f64,bool,str,str,str,str,str,str,str
"""SAGGSAGGSAGGKVTIVVENIKVFGEDGKL…","""SAGGSAGGSAGGKVTIVVENIKVFGEDGKL…","""E33R""",-0.088899,false,"""megascale""","""G3DSA:2.40.50.140""","""G3DSA:2.40.50.140;G3DSA:6.20.3…","""G3DSA:2""","""40""","""50""","""140"""
"""SAGKMTGIVKWFNADKGFGFITPDDGSKDV…","""SAGKMTGIVKWFNADKGFGPITPDDGSKDV…","""F20P""",-0.216443,false,"""megascale""","""G3DSA:3.30.170.10""","""G3DSA:3.30.170.10""","""G3DSA:3""","""30""","""170""","""10"""
"""SAGGMIINNLKLIREKKKISQSELAALLES…","""SAGGMIIKNLKLIREKKKISQSELAALLES…","""N8K""",-0.262815,false,"""megascale""","""G3DSA:1.10.30.10""","""G3DSA:1.10.30.10""","""G3DSA:1""","""10""","""30""","""10"""
"""SAGGSAGGSAQGDIVVALYPYDGIHPDDLS…","""SAGGSAGGSAQGDIVVALYPIDGIHPDDLS…","""Y21I:Y64R""",-0.39773,false,"""megascale""","""G3DSA:2.30.30.140""","""G3DSA:2.30.30.140""","""G3DSA:2""","""30""","""30""","""140"""
"""SAGKMTGIVKWFNADKGFGFITPDDGSKDV…","""SAGKMTGIVKWFNADKGFGFITPGDGSKDV…","""D24G""",-0.145054,false,"""megascale""","""G3DSA:3.30.170.10""","""G3DSA:3.30.170.10""","""G3DSA:3""","""30""","""170""","""10"""
…,…,…,…,…,…,…,…,…,…,…,…
"""SAGGSDAPDEFRDPLMDDLMTDPVRLPSGT…","""SAGGSDAPDEFRDPLMDTLMTDPVRLPSGT…","""T18D:D33S""",0.660616,true,"""megascale""","""G3DSA:2.30.42.10""","""G3DSA:1.20.1160.20;G3DSA:2.30.…","""G3DSA:2""","""30""","""42""","""10"""
"""SAGGSGPGLTDLFKTEKAAVKKMAKAIMAD…","""SAGGSGPGLTDLFKTEKAAVKKMAKAIMAD…","""Y38P:Y73E""",0.395344,true,"""megascale""","""G3DSA:1.10.30.10""","""G3DSA:1.10.30.10""","""G3DSA:1""","""10""","""30""","""10"""
"""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRV…","""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRE…","""E30V:Y50G""",0.067857,true,"""megascale""","""G3DSA:1.10.30.10""","""G3DSA:1.10.30.10""","""G3DSA:1""","""10""","""30""","""10"""


In [4]:
df_255.filter(pl.col("original_seq_full").is_in(holdout_sequences))

NameError: name 'df_255' is not defined